# 03 — Fetch Global Database of Events, Language, and Tone (GDELT) Headlines
**GeoSentinel Terminal (VARTA) · Team 7 Lambda · SP2026**

Fetches geopolitical news headlines from the Global Database of Events, Language, and Tone (GDELT) Project
using the GDELT Document 2.0 Application Programming Interface (free, no key required).
Filters for supply chain, critical minerals, and geopolitical risk terms.

Outputs: `data/processed/gdelt.parquet`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

In [ ]:
import requests
import pandas as pd
import polars as pl
from config import GDELT_FILTER_TERMS, DATE_TRAIN_START, DATE_TRAIN_END, DATA_PROC
from src.utils import log, save_parquet

log.info(f"Filter terms ({len(GDELT_FILTER_TERMS)}): {GDELT_FILTER_TERMS}")

In [ ]:
# ── Query GDELT Document 2.0 API ──────────────────────────────────────────────
# Strategy: sample across years (2010-2024) with top priority terms only.
# GDELT DOC 2.0 returns max 250 articles per query; no bulk historical access.
# Rate limit: 1 req/s — we add 2s delays and exponential backoff on 429.
import time

BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"

# Sample 4 years per decade to get temporal coverage without hitting rate limits
SAMPLE_YEARS = [2011, 2013, 2015, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

# Use top 6 most signal-rich terms to keep total requests manageable
PRIORITY_TERMS = [
    "geopolitical risk supply chain",
    "semiconductor supply chain disruption",
    "critical minerals rare earth",
    "trade war sanctions",
    "Ukraine Russia energy",
    "Taiwan semiconductor chip",
]

frames = []
total_requests = 0

for year in SAMPLE_YEARS:
    start_dt = f"{year}0101000000"
    end_dt   = f"{year}1231235959"
    for term in PRIORITY_TERMS:
        params = {
            "query":         term,
            "mode":          "artlist",
            "maxrecords":    "250",
            "startdatetime": start_dt,
            "enddatetime":   end_dt,
            "format":        "json",
        }
        for attempt in range(3):
            try:
                resp = requests.get(BASE_URL, params=params, timeout=30)
                if resp.status_code == 429:
                    wait = 30 * (attempt + 1)
                    log.warning(f"  Rate limited — waiting {wait}s...")
                    time.sleep(wait)
                    continue
                resp.raise_for_status()
                articles = resp.json().get("articles", [])
                if articles:
                    df_t = pd.DataFrame(articles)
                    available_cols = [c for c in ["seendate", "title", "tone", "url"] if c in df_t.columns]
                    df_t = df_t[available_cols]
                    if "seendate" in df_t.columns:
                        df_t = df_t.rename(columns={"seendate": "date", "title": "headline"})
                    df_t["date"] = pd.to_datetime(df_t["date"], format="%Y%m%dT%H%M%SZ", errors="coerce")
                    df_t["query_term"] = term
                    frames.append(df_t)
                    log.info(f"  {year} '{term[:30]}': {len(articles)} articles")
                total_requests += 1
                time.sleep(2)  # respect rate limit
                break
            except Exception as e:
                log.warning(f"  {year} '{term[:30]}' attempt {attempt+1} failed: {e}")
                time.sleep(5)

if not frames:
    raise RuntimeError("No articles fetched — check internet connection and GDELT API")

combined = pd.concat(frames, ignore_index=True)
# Ensure required columns exist
for col in ["headline", "tone", "url", "query_term"]:
    if col not in combined.columns:
        combined[col] = None

combined = combined.drop_duplicates(subset=["url"])
df = pl.from_pandas(combined[["date", "headline", "tone", "url", "query_term"]])
print(f"Total unique articles: {len(df):,} across {len(SAMPLE_YEARS)} years")
df.head()

In [ ]:
# ── Validate ──────────────────────────────────────────────────────────────────
passed = True

if len(df) < 100:
    log.warning(f"Only {len(df)} articles — may be insufficient for scoring")
    passed = False
else:
    log.info(f"✓ {len(df):,} articles")

null_headlines = df["headline"].null_count()
if null_headlines > 0:
    log.warning(f"{null_headlines} null headlines — will be dropped before scoring")
    df = df.filter(pl.col("headline").is_not_null())

# Tone distribution
print("Tone distribution (GDELT tone: negative = negative sentiment, positive = positive):")
print(df.select(pl.col("tone").cast(pl.Float64)).describe())

# Articles per query term
print("\nArticles per query term:")
print(df.group_by("query_term").agg(pl.len().alias("count")).sort("count", descending=True))

print(f"\nValidation passed: {passed}")

In [ ]:
# ── Save — add llm_score placeholder column (filled in notebook 06) ───────────
df = df.with_columns(pl.lit(None).cast(pl.Float64).alias("llm_score"))
assert passed, "Validation failed — check warnings above"
save_parquet(df, DATA_PROC / "gdelt.parquet", "GDELT headlines")
print("Saved → data/processed/gdelt.parquet")

In [ ]:
# ── Tone distribution chart ────────────────────────────────────────────────────
import plotly.express as px

df_plot = df.with_columns(pl.col("tone").cast(pl.Float64)).to_pandas()
fig = px.histogram(
    df_plot, x="tone", nbins=50, color="query_term",
    title="Global Database of Events Headline Tone Distribution by Query Term",
    labels={"tone": "GDELT Tone Score (negative = negative sentiment)"},
    template="plotly_dark",
)
fig.show()